# Convolutional Neural Networks
Now that we've gone through neural networks we can focus on a topic of particular interest for us as a medical imaging team, convolutional neural networks (CNNs).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Module
import torch.optim as optim
from torch.optim import Optimizer
from torch.utils.data import Dataset, DataLoader, random_split
from torch import Tensor
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import time
import pandas as pd
from sklearn.metrics import confusion_matrix
type LossFN = Union[Module]
device = (
    "cuda" # For NVIDIA GPUs
    if torch.cuda.is_available()
    else "mps" # For Apple devies
    if torch.backends.mps.is_available()
    else "cpu"
)

### Hand-Writing Recognition
Hand-writing recognition is a classic problem neural network that involves recognising numbers from people's hand-writing. This is done on the MNIST dataset, a dataset created from the handwriting of American census bureau employees and highschool students. QMNIST is a restored version of the dataset that can actually be easily downloaded from Torch. The calls below do our job for us and get us the data without requiring us to download or make a dataset manually.

Additionally a series of digits has been plotted to allow you to examine what the dataset and its labels look like. Each digit is restricted to a grayscale 28x28 space with an associated label.

In [ ]:
# Remember this place for later
preprocessing = transforms.ToTensor()

# Download training data from torch
training_data = datasets.QMNIST(
    root="data",
    train=True,
    download=True,
    transform=preprocessing,
)

# Download test data from torch
test_data = datasets.QMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

# Create dataloaders
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# Display the data
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(label)
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

### **(Question 1)** Model
Our first goal is to make a basic neural network for our dataset. This involves creating a module like before. To create this focus on creating blocks of convolutional layers, activation layers and pooling layers. Some useful functions to try out are:
- `nn.Linear`
- `nn.Conv2d`
- `nn.ReLu`
- `MaxPool2d`

**Build a basic neural network**. Focus on it working at first, we can improve this later.

In [ ]:
class QMNISTNet(Module):
    def __init__(self):
        """Layers for our handwritting model"""
        super().__init__()
        # Your Code Goes Here
        pass
    def forward(self, x: Tensor) -> Tensor:
        """Give handwritting prediction"""
        return self.model(x)

### **(Question 2)** Training
From our previous week we saw how you would go about training and testing a model. Our first order of business is to make it less messy. A nice way of doing this is by declaring seperate function for `train` and `test`. This will allow us to take parameters such as the dataloader, model, loss function, and optimiser as arguments which can be useful if we want to test multiple different things. Similar to last time you need to iterate through your data, forward pass, and back pass. Look at the previous week if you need some help.

**Create a function that takes in a dataloder, a model, a loss function, an optimiser, and trains the network**. This function will be called every epoch so don't worry about making a loop for the amount of epochs here. 

In [ ]:
def train(dataloader: DataLoader, model: Module, loss_fn: LossFN, optimiser: Optimizer):
    """Train the model using the training set"""
    # Your Code Goes Here
    pass


### **(Question 3)** Testing
Once we have finished running training we can now test the model against our testing dataset. This will provide us with a look into its current loss and accuracy. This will be more different to our previous example as we now want to calculate both the loss and accuracy after running the test function. This will be done similar to training however rather than backpropagating we store our loss and accuracy, allowing us to print them at the end of the epoch.

**Create a test function that computes the loss and accuracy for data in the test dataloader**. Make sure to divide the loss and accuracy at the end. To predict our accuracy you can calculate from a prediction `pred`, and a label `y` the result to be:
```python
res = pred.argmax(dim=1, keepdim=True)
acc += res.eq(y.view_as(res)).sum().item()
```

In [ ]:
def test(dataloader: DataLoader, model: Module, loss_fn: LossFN):
    """Test the model to see how it compares against the test portion of the dataset"""
    loss = 0
    acc = 0
    # Your Code Goes Here
    print(f"Accuracy: {(acc):>0.4f}%, Avg loss: {loss:>8f}")
    (acc, loss)

### **(Question 4)** Loss & Optimiser
The final step before we can run our model is choosing a loss function and optimiser. For now you can just enter the previously used ones, however be sure to change these later to see how the accuracy changes given these variables.

**Choose a loss function and optimiser**.

In [ ]:
model = QMNISTNet()
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=1e-3)

### Results
Given that you have done the previous steps correctly you should be able to run the below code as to train the model for yourself. This code runs 10 epochs before finishing. If you are willing to spend more time training be sure to modify the amount of epochs. Don't expect this to be super accurate on the first try as we are still learning.

In [ ]:
epochs = 10
results = []
for t in range(epochs):
    print(f"Epoch {t+1}/{epochs}: ", end="")
    train(train_dataloader, model, loss_fn, optimiser)
    results.append(test(test_dataloader, model, loss_fn))
print("Done!")

### **(Question 5)** Confusion Matrix
A confusion matrix allows for analysis of the models effectiveness over each category of the labels. This can allow us to find aspects of our model which are lacking

**Implement the test function displaying a confusion matrix**. To achieve this redo the test function except add a call to `confusion_matrix` and display it.

In [ ]:
def test(dataloader: DataLoader, model: Module, loss_fn: LossFN):
    """Test the model to see how it compares against the test portion of the dataset"""
    loss = 0
    acc = 0
    # Your Code Goes Here
    print(f"Accuracy: {(acc):>0.4f}%, Avg loss: {loss:>8f}")
    print(cm)
    return acc, loss, cm

### **(Question 6)** Preprocessing
Now that you have done all the previous steps you should have a model that ranges from ok to pretty decent. However, one thing that may be holding you back is the lack of preprocessing. So what is preprocessing and how can we use it?

Preprocessing is the idea of modifying your data before you use it in a network. An example you have already done is back when you encoded male and female as 0 and 1. Another example can also be filling in data points that you can't find with a mean/median. With images this approach is different, rather we want to normalise the data, do rotations, change pixels and more.

The goal with preprocessing in general is to make our dataset more generalisable so that it not only can have an easier time training, but be able to be ran on new data and get a more accurate result. Below is an example of how we can change the preprocessing steps in our code. This creates a pipeline that starts by rotating before turning our input to a tensor.
```python
preprocessing = transforms.Compose([
    transforms.RandomRotation((0, 360)),
    transforms.ToTensor(),
])
```

**Modify the preprocessing step to be more advanced**. This step can be found in the place you downloaded the dataset. Be sure to experiment with different things not just rotations.

### **(Question 7)** Improve your Model
Congratulations you have trained a neural network. Most likely the results from this will be less than satisfactory, with it being possibly lower than 50%. Now this is only the start of your journey, every single variable whether your *loss function*, *optimiser (and its learning rate)*, *extracted features*, *model layers*, and more can be changed as to create a better model. This process if called **hyperparameter tuning** and is a large part of machine learning. If you ever want to get better at ML you must be willing to spend the time the improve your model. Therefore the last task is:

**Improve your model to have a better accuracy**. Tune respective parameters as to produce the best result possible. This will require some research.

In [ ]:
# If you want to pretify your loss and accuracy graphs here:
def plot_loss_acc(loss: List[int], acc: List[int]):
    """Helper to plot loss and accuracy if you store them in a list"""
    fig, (ax0, ax1) = plt.subplots(1,2,figsize=(16,5))
    ax0.plot(acc, 's-')
    ax0.set_title(f'Final test accuracy {acc[-1]:.2f}%')
    ax0.set_ylabel('Accuracy%')
    ax0.set_xlabel('Epochs')
    ax1.plot(loss, 's-')
    ax1.set_title(f'Final test loss {loss[-1]:.2f}')
    ax1.set_ylabel('Loss')
    ax1.set_xlabel('Epochs')